# 03.单变量初筛

合并基础统计量、PSI、KS，输出 3A score 全保留和 3B score 筛选版本。

In [ ]:
import os
import sys
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "output"
SRC_DIR = PROJECT_ROOT / "src"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(SRC_DIR))

from 函数代码包 import *

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: "%.6f" % x)

customer_type = "demo_existing_customer"
target_type = "y2"   # 可切换为 "y1"

join_keys = ["id", "cell", "name", "apply_date"]
keep_cols = ["month", "flag"]
target = "y"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("customer_type:", customer_type)
print("target_type:", target_type)

In [ ]:
stat_df = pd.read_csv(OUTPUT_DIR / f"1.基础统计量_{customer_type}_{target_type}.csv")
psi_df = pd.read_csv(OUTPUT_DIR / f"1.PSI结果_train_valid_oot_{customer_type}_{target_type}.csv")
ks_df = pd.read_csv(OUTPUT_DIR / f"2.KS结果_{customer_type}_{target_type}.csv")
data_df = pd.read_csv(OUTPUT_DIR / f"1.建模宽表_{customer_type}_{target_type}.csv")

delete_file = DATA_DIR / "delete_fields.xlsx"
if delete_file.exists():
    delete_df = pd.read_excel(delete_file)
    delete_vars = set(delete_df["enname"].dropna().astype(str))
else:
    delete_vars = set()

print(stat_df.shape, psi_df.shape, ks_df.shape, data_df.shape)

In [ ]:
stat_statement = "(stat_df['same_rate'] < 0.95) & (stat_df['match_rate'] > 0.05)"
psi_statement = "(psi_df['psi_max'] < 0.1) & (psi_df['psi_max'] >= 0)"
ks_statement = "(ks_df['ks_min'] > 0.02) & (ks_df['ks_all'] > 0.02) & (ks_df['ks_cv'] <= 0.4)"

start_cols = ["var", "dtype"]

cur_cat = psi_df[psi_df["dtype"] == "类别型"][start_cols]
cur_num = psi_df[psi_df["dtype"] == "数值型"][start_cols]
cur_score = psi_df[psi_df["dtype"] == "评分"][start_cols]

stat_filtered = stat_df[eval(stat_statement)][["var", "same_rate", "match_rate"]]
psi_filtered = psi_df[eval(psi_statement)][["var", "psi_max", "psi_avg"]]
ks_filtered = ks_df[eval(ks_statement)][["var", "ks_all", "ks_avg", "ks_min", "ks_std", "ks_cv", "ks_range"]]

# 控制变量量级
ks_filtered = ks_filtered.sort_values(["ks_all", "ks_avg"], ascending=False).head(1500)

In [ ]:
def build_screen_result(score_mode="3A"):
    cat = cur_cat.merge(stat_filtered, on="var", how="inner").merge(psi_filtered, on="var", how="inner")
    num = cur_num.merge(stat_filtered, on="var", how="inner").merge(psi_filtered, on="var", how="inner").merge(ks_filtered, on="var", how="inner")

    score = cur_score.merge(stat_df[["var", "same_rate", "match_rate"]], on="var", how="left")
    score = score.merge(psi_df[["var", "psi_max", "psi_avg"]], on="var", how="left")
    score = score.merge(ks_df[["var", "ks_all", "ks_avg", "ks_min", "ks_std", "ks_cv", "ks_range"]], on="var", how="left")

    if score_mode == "3B":
        score = score[(score["psi_max"] < 0.1) & (score["psi_max"] >= 0) & (score["ks_all"] > 0.02)]

    result = pd.concat([num, cat, score], axis=0, ignore_index=True)
    result = result[~result["var"].isin(delete_vars)].copy()
    return result

fea_3a = build_screen_result("3A")
fea_3b = build_screen_result("3B")

fea_3a.to_csv(OUTPUT_DIR / f"3A.单变量初筛结果_score全保留_{customer_type}_{target_type}.csv", index=False)
fea_3b.to_csv(OUTPUT_DIR / f"3B.单变量初筛结果_score筛选_{customer_type}_{target_type}.csv", index=False)

print(fea_3a.shape, fea_3b.shape)
display(fea_3a.head())

In [ ]:
for tag, fea in [("3A", fea_3a), ("3B", fea_3b)]:
    cols = list(dict.fromkeys(join_keys + keep_cols + [target, "y1", "y2"] + fea["var"].tolist()))
    cols = [c for c in cols if c in data_df.columns]

    out = data_df[cols].copy().fillna(-999)

    if tag == "3A":
        file_name = f"3A.单变量初筛数据_score全保留_{customer_type}_{target_type}.csv"
    else:
        file_name = f"3B.单变量初筛数据_score筛选_{customer_type}_{target_type}.csv"

    out.to_csv(OUTPUT_DIR / file_name, index=False)
    print(tag, out.shape, file_name)